In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Configura tu conexión (Descomenta la que uses)
# -------------------------------------------------------------
# Para PostgreSQL:
engine = create_engine('DATABASE_URL')  # Reemplaza con tu URL real de conexión

# -------------------------------------------------------------

# 2. Ruta de tu archivo CSV
csv_file = "../data/processed/base_users.csv" 

print("📖 Leyendo el archivo CSV...")

# Leemos el CSV en bloques (chunks) por si el archivo es gigantesco y no cabe en RAM
# 'chunksize' define cuántas filas procesa a la vez. 10,000 o 50,000 es un buen equilibrio.
chunk_size = 20000 
total_rows = 0

# NOTA: Si sospechas que usa comas, cambia sep='\t' por sep=',' o sep=';'
for i, chunk in enumerate(pd.read_csv(csv_file, sep=',', chunksize=chunk_size)): 
    
    # TRUCO CLAVE: Elimina espacios ocultos al principio o final de los nombres de columnas
    chunk.columns = chunk.columns.str.strip()
    
    # Si quieres verificar qué columnas ha detectado en el primer bloque, descontinúa la línea de abajo:
    # if i == 0: print("Columnas detectadas:", chunk.columns.tolist())

    try:
            # 1. Columnas booleanas: Las aseguramos pasándolas primero por un mapeo flexible
            columnas_bool = ['email_verificado', 'paso_3d_secure']
            
            for col in columnas_bool:
                # Convertimos a string, limpiamos espacios y pasamos a minúsculas para comparar bien
                s = chunk[col].astype(str).str.strip().str.lower()
                # Mapeamos los valores típicos a True/False reales. Lo que no coincida (como vacíos o NaN) será False.
                chunk[col] = s.isin(['true', '1', 'yes', 'si', 's'])

            chunk['bloqueado'] = False # siempre no bloqueado

            # 2. Columna numérica: Esta se queda como entero
            chunk['dias_antiguedad_cuenta'] = pd.to_numeric(chunk['dias_antiguedad_cuenta'], errors='coerce').fillna(0).astype(int)

    except KeyError as e:
        print(f"❌ Error: No se encontró la columna {e}.")
        print(f"Las columnas reales que Pandas ve son: {chunk.columns.tolist()}")
        break
    
    print(f"🚀 Subiendo bloque {i+1} ({len(chunk)} filas)...")
    
    chunk.to_sql(
        name='clientes', 
        con=engine, 
        if_exists='append',
        index=False,
        method='multi'
    )
    
    total_rows += len(chunk)

print(f"✅ ¡Proceso terminado! Total subido: {total_rows} usuarios.")

📖 Leyendo el archivo CSV...
🚀 Subiendo bloque 1 (9999 filas)...
✅ ¡Proceso terminado! Total subido: 9999 usuarios.


In [5]:
csv_file = "../data/processed/base_users.csv" 

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
engine = create_engine('DATABASE_URL')
df_usuarios = pd.read_sql_table('clientes', con=engine)

In [4]:
df_usuarios

,id_usuario,nombre,apellido,dni,email,email_verificado,dias_antiguedad_cuenta,pais_emision,paso_3d_secure,bloqueado,intentos,intentos_expires_at,ultima_actualizacion
